In [33]:
import pandas as pd
import numpy as np

In [34]:
live_avg_mega = pd.read_csv("../samples/mega_processed/live_avg.csv")
live_avg_ollama = pd.read_csv("../samples/ollama_processed/live_avg.csv")

In [35]:
def weights_builder(df):
    gpu_util = df['GPU Util']
    mem_utill = df['Mem Util']
    rt = df["Response Time"]
    rt_avg = np.average(rt)
    gpu_std = np.std(gpu_util)
    mem_std = np.std(mem_utill)
    w1 = 1/gpu_std
    w2 = 1/mem_std
    w3 = -np.exp(rt_avg/(25/np.log(2)))
    weights = [w1, w2, w3]
    weights_norm = weights / np.sum(np.abs(weights)) 
    return weights_norm


In [36]:
live_avg_ollama

,GPU Util,Power Draw,GPU Temp,GPU Clock,Mem Used,Mem Util,Time,Time Delta,Iter,Response Time,Eval Rate
0,0.000000,23.740000,22.000000,1.350000e+08,9.437184e+06,0.000000,-2.208933e+18,1.916667,6.5,2.606603,92.848743
1,0.000000,23.740000,22.000000,1.350000e+08,9.437184e+06,0.000000,-2.208933e+18,3.666667,2.0,0.522210,82.672835
2,0.000000,23.740000,22.000000,1.350000e+08,9.437184e+06,0.000000,-2.208933e+18,5.307692,7.0,2.582972,82.589487
3,0.000000,23.740000,22.000000,1.350000e+08,9.437184e+06,0.000000,-2.208933e+18,7.200000,3.0,0.955865,82.795251
4,0.033333,153.831667,27.833333,1.172500e+09,6.665798e+09,0.030000,-2.208933e+18,8.500000,3.5,1.145166,82.972630
...,...,...,...,...,...,...,...,...,...,...,...
64,0.805714,188.162679,43.589286,1.380000e+09,6.669992e+09,0.542321,-2.208932e+18,504.142857,28.5,13.239929,80.305746
65,0.810000,187.830000,44.000000,1.380000e+09,6.669992e+09,0.530000,-2.208932e+18,514.178571,14.5,6.724487,80.335888
66,0.491379,180.018966,43.034483,1.380000e+09,6.669992e+09,0.288621,-2.208932e+18,521.862069,15.0,9.312371,82.007438
67,0.579000,178.675000,43.300000,1.380000e+09,6.669992e+09,0.355000,-2.208932e+18,531.450000,10.5,7.641057,80.087957


In [37]:
live_v = pd.read_csv("../samples/ollama/live_metrics.csv")
live_a = pd.read_csv("../samples/mega/live_metrics.csv")

In [38]:
verbose_v = pd.read_csv("../samples/ollama/verbose_statements.csv")
verbose_a = pd.read_csv("../samples/mega/verbose_statements.csv")

In [39]:
live_v = live_v.dropna()
live_a = live_a.dropna()
verbose_v = verbose_v.dropna()
verbose_a = verbose_a.dropna()

In [40]:
weights_a = weights_builder(live_avg_mega)
weights_v = weights_builder(live_avg_ollama)

In [41]:
weights_a, weights_v

(array([ 0.32220293,  0.61772018, -0.0600769 ]),
 array([ 0.3641675 ,  0.53263736, -0.10319514]))

In [42]:
avg_v_gpu = np.average(live_v['GPU Util'])
avg_v_mem = np.average(live_v['Mem Util'])
avg_a_gpu = np.average(live_a['GPU Util'])
avg_a_mem = np.average(live_a['Mem Util'])
avg_v_rt = np.average(verbose_v['total_duration'])
avg_a_rt = np.average(verbose_a['total_duration'])

In [43]:
averages_a = [avg_a_gpu, avg_a_mem, avg_a_rt]
averages_v = [avg_v_gpu, avg_v_mem, avg_v_rt]

In [44]:
score_v = np.dot(weights_v, averages_v)
score_a = np.dot(weights_a, averages_a)

In [45]:
score_a, score_v

(-215494071.6176475, -810410728.874979)